In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2,math
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision.utils as vutils
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
workers = 2
batch_size = 300
nz = 100
num_epochs = 5
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize(100),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
#fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
train = torchvision.datasets.ImageFolder(root=f'/kaggle/input/d/mafiosoquasar/detect-ai-vs-human-generated-images/train',transform=transform)

dataloader = torch.utils.data.DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=2)

fsc_submission=numpy.array(pandas.read_csv("/kaggle/input/detect-ai-vs-human-generated-images/test.csv")['id'])
#print(len(dataloader),len(testloader))
device = torch.device("cuda" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.swin_v2_b(weights=torchvision.models.swin_transformer.Swin_V2_B_Weights.IMAGENET1K_V1)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1024, 1)
        )
        
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [6]:
class customiser:
    def __init__(self):
        print('loss_optimizer')
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr, betas=(beta1, 0.999))
        self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=0.86)
        
    def train(self, EFF_NET):
        print('train')
        i_w=0
        while True:
            print('hi')
            z_,z,z_w=0,0,{}
            correct=0
            for i, data in enumerate(dataloader, 0):
                print(0)
                label = data[1].to(device).float()
                raw_output = EFF_NET(data[0].to(device)).float().view(-1)
                print(1)
                err_real = self.criterion(raw_output, data[1].float().to(device))
                self.optimizer.zero_grad()
                err_real.backward()
                self.optimizer.step()
                print(err_real.item())
                
            print(f"EPOCH: {z_k} LOSS_FSC: {err_real.item()}")
            if err_real.item()<=0.02:
                torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()} {z_k}.pth')# {z_add}{acc}
            if z_k==100:
                break
            z_k+=1
            
    def submission_div(self, EFF_NET):
        sub = ([0]*len(fsc_submission))
        file_ = ['/kaggle/input/detect-ai-vs-human-generated-images-mode-2/0.0005000190576538444 10.pth']
        EFF_NET.load_state_dict(torch.load(f"{file_[0]}", weights_only=False,
                                           map_location=torch.device('cpu')))
        for i in range(0, len(fsc_submission), 10):
            img = torch.Tensor(numpy.array([transform(Image.open(f'/kaggle/input/d/mafiosoquasar/detect-ai-vs-human-generated-images/test/{fsc_submission[s].split("/")[1]}')).reshape((3, 100, 100)) for s in range(i, i+10)])).float().to(device)    
            z_arr = EFF_NET(img).sigmoid().cpu().detach().numpy().reshape(-1)
            for j,n in zip(z_arr, range(i, i+10)):
                if j>0.5:
                    sub[n] = 1
                else:
                    sub[n ]= 0
        
        submission=pandas.DataFrame({'id' : fsc_submission, 'label' : numpy.array(sub)})
        pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
        pandas.DataFrame(submission)

In [7]:
EFF_NET = EffnetModel().float()
EFF_NET = nn.DataParallel(EFF_NET).to(device)

Downloading: "https://download.pytorch.org/models/swin_v2_b-781e5279.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_b-781e5279.pth
100%|██████████| 336M/336M [00:02<00:00, 165MB/s]


In [8]:
cust = customiser()
cust.submission_div(EFF_NET)
#cust.train(EFF_NET)

loss_optimizer
